# AMML Ranking — финальное решение

**Финальная конфигурация:** Experiment 21 Best Pair — 39% рангов Experiment 18 Champion + 61% рангов Jina Reranker v2 LoRA из Experiment 20.

Ноутбук содержит:
- подготовку исходных `train.csv`, `test.csv`, `sample_submission.csv`;
- воспроизведение финального ансамбля с зафиксированными весами;
- полный код пятифолдовочного обучения Jina v2 + LoRA для повторного обучения при необходимости.

По умолчанию используется проверенный кэш OOF/test-оценок, встроенный в ноутбук. Поэтому стандартный запуск создаёт итоговый `submission.csv` без повторного GPU-обучения. Для полного повторного обучения установите `USE_CACHED_FINAL_SCORES = False`.

**Вход:** только три файла соревнования. Оценки Champion Experiment 18 и LoRA Experiment 20 встроены в ноутбук и выравниваются по `id`.

**Среда исходного обучения Jina LoRA:** Linux, Google Colab, NVIDIA Tesla T4 (14.56 GiB), Python 3.13.16, PyTorch 2.14.1+cu130, Transformers 5.22.1, PEFT 0.21.2.


## 1. Импорты и параметры финального решения

В ноутбуке нет подбора весов, перебора моделей или промежуточной аналитики. Параметры финального ансамбля зафиксированы по результатам Experiment 21.


In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys, re, io, gc, json, math, gzip, base64, random, shutil, zipfile, inspect, warnings, platform, subprocess
from pathlib import Path
from importlib.metadata import version as package_version, PackageNotFoundError

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from IPython.display import display

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
except ImportError:
    print('PyTorch is not installed; cached-score mode can still run, full retraining cannot.')

warnings.filterwarnings("ignore")

USE_CACHED_FINAL_SCORES = True  # False = full 5-fold GPU retraining
SEED = 42
N_SPLITS = 5
TOP_K = 10

MODEL_ID = "jinaai/jina-reranker-v2-base-multilingual"
MAX_LENGTH = 512
NUM_EPOCHS = 1
PAIR_LIMIT_PER_QUERY = 20
PAIR_BUDGET_PER_FOLD = 6000
TRAIN_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 38
LOGGING_STEPS = 50
SAVE_STEPS = 250

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["mixer.out_proj", "fc1", "fc2"]

FINAL_W_EXP18 = 0.39
FINAL_W_LORA = 0.61
assert abs(FINAL_W_EXP18 + FINAL_W_LORA - 1.0) < 1e-12

OUTPUT_DIR = Path("/content/solution_artifacts") if Path("/content").is_dir() else Path.cwd() / "solution_artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("System:", platform.system())
print("Platform:", platform.platform())
print("Python:", sys.version.split()[0])
for package in ["numpy", "pandas", "scikit-learn", "torch", "transformers", "peft", "einops"]:
    try:
        print(f"{package}: {package_version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed in this runtime")


## 2. Загрузка исходных данных

Поддерживаются:
- отдельные `train.csv`, `test.csv`, `sample_submission.csv`;
- ZIP соревнования, содержащий эти три файла;
- Kaggle Input или загрузка файлов в Colab.

Если обнаружено несколько файлов с одинаковыми именами, ноутбук останавливается и показывает пути, вместо случайного выбора.


In [ ]:
MANUAL_PATHS = {"train": None, "test": None, "sample_submission": None}

SEARCH_ROOTS = []
for candidate in [Path("/content"), Path("/kaggle/input"), Path.cwd(), Path("/mnt/data")]:
    if candidate.exists() and candidate not in SEARCH_ROOTS:
        SEARCH_ROOTS.append(candidate)

EXCLUDED_DIR_NAMES = {"solution_artifacts", "sample_data", "__pycache__", ".cache", ".ipynb_checkpoints"}

def is_excluded(path):
    return any(part in EXCLUDED_DIR_NAMES for part in path.parts)

def find_named_files(filename):
    found, seen = [], set()
    for root in SEARCH_ROOTS:
        try:
            for path in root.rglob(filename):
                if not path.is_file() or is_excluded(path):
                    continue
                resolved = path.resolve()
                if resolved not in seen:
                    seen.add(resolved)
                    found.append(resolved)
        except (PermissionError, OSError):
            continue
    return sorted(found, key=lambda p: (len(str(p)), str(p)))

def extract_competition_zip():
    zip_candidates, seen = [], set()
    for root in SEARCH_ROOTS:
        try:
            for path in root.rglob("*.zip"):
                if not path.is_file() or is_excluded(path):
                    continue
                resolved = path.resolve()
                if resolved in seen:
                    continue
                seen.add(resolved)
                try:
                    with zipfile.ZipFile(resolved) as archive:
                        basenames = {Path(name).name for name in archive.namelist() if not name.endswith("/")}
                        if {"train.csv", "test.csv", "sample_submission.csv"}.issubset(basenames):
                            zip_candidates.append(resolved)
                except (zipfile.BadZipFile, OSError):
                    continue
        except (PermissionError, OSError):
            continue

    if len(zip_candidates) == 0:
        return None
    if len(zip_candidates) > 1:
        raise FileNotFoundError(
            "Найдено несколько ZIP соревнования. Укажи один архив вручную:\n"
            + "\n".join(map(str, zip_candidates[:20]))
        )

    archive_path = zip_candidates[0]
    extract_dir = OUTPUT_DIR / "competition_data"
    extract_dir.mkdir(parents=True, exist_ok=True)
    wanted = {"train.csv", "test.csv", "sample_submission.csv"}
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.namelist():
            basename = Path(member).name
            if basename in wanted and not member.endswith("/"):
                destination = extract_dir / basename
                with archive.open(member) as src, open(destination, "wb") as dst:
                    shutil.copyfileobj(src, dst)
    print("Competition ZIP:", archive_path)
    return extract_dir

def resolve_input_path(key, filename):
    manual = MANUAL_PATHS.get(key)
    if manual:
        path = Path(manual).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f"MANUAL_PATHS['{key}'] не найден: {path}")
        return path.resolve()

    candidates = find_named_files(filename)
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        parents = {p.parent for p in candidates}
        if len(parents) == 1:
            return candidates[0]
        raise FileNotFoundError(
            f"Найдено несколько {filename}. Укажи точный путь в MANUAL_PATHS['{key}']:\n"
            + "\n".join(map(str, candidates[:20]))
        )
    return None

TRAIN_PATH = resolve_input_path("train", "train.csv")
TEST_PATH = resolve_input_path("test", "test.csv")
SAMPLE_PATH = resolve_input_path("sample_submission", "sample_submission.csv")

if not all([TRAIN_PATH, TEST_PATH, SAMPLE_PATH]):
    extracted_dir = extract_competition_zip()
    if extracted_dir is not None:
        TRAIN_PATH = TRAIN_PATH or extracted_dir / "train.csv"
        TEST_PATH = TEST_PATH or extracted_dir / "test.csv"
        SAMPLE_PATH = SAMPLE_PATH or extracted_dir / "sample_submission.csv"

if not all([TRAIN_PATH, TEST_PATH, SAMPLE_PATH]):
    raise FileNotFoundError(
        "Не найдены все три входных файла. Загрузи train.csv, test.csv и sample_submission.csv "
        "в /content или подключи Kaggle Input. При нескольких копиях укажи MANUAL_PATHS."
    )

print("TRAIN:", TRAIN_PATH)
print("TEST :", TEST_PATH)
print("SAMPLE:", SAMPLE_PATH)

def read_csv_strict(path, role):
    errors = []
    for engine in ("c", "python"):
        try:
            kwargs = {
                "encoding": "utf-8-sig",
                "engine": engine,
                "dtype": {"id": "string"},
                "on_bad_lines": "error",
            }
            if engine == "c":
                kwargs["low_memory"] = False
            frame = pd.read_csv(path, **kwargs)
            print(f"{role}: прочитан ({engine}), shape={frame.shape}")
            return frame
        except Exception as exc:
            errors.append(f"{engine}: {type(exc).__name__}: {exc}")
    raise RuntimeError(
        f"Не удалось целиком прочитать {role}: {path}\n" + "\n".join(errors)
        + "\nСтроки не пропускались. Проверь целостность CSV."
    )


In [ ]:
train = read_csv_strict(TRAIN_PATH, "TRAIN")
test = read_csv_strict(TEST_PATH, "TEST")
sample_submission = read_csv_strict(SAMPLE_PATH, "SAMPLE_SUBMISSION")

REQUIRED_TRAIN = {"id", "query_id", "document_id", "query", "title", "text", "relevance"}
REQUIRED_TEST = REQUIRED_TRAIN - {"relevance"}

if not REQUIRED_TRAIN.issubset(train.columns):
    raise KeyError(f"В train.csv отсутствуют колонки: {sorted(REQUIRED_TRAIN - set(train.columns))}")
if not REQUIRED_TEST.issubset(test.columns):
    raise KeyError(f"В test.csv отсутствуют колонки: {sorted(REQUIRED_TEST - set(test.columns))}")
if not {"id", "prediction"}.issubset(sample_submission.columns):
    raise KeyError("В sample_submission.csv ожидаются колонки id,prediction.")

for name, frame in [("train", train), ("test", test), ("sample_submission", sample_submission)]:
    frame["id"] = frame["id"].astype(str)
    if not frame["id"].is_unique:
        raise ValueError(f"В {name} повторяются id.")

for frame in [train, test]:
    for col in ["query_id", "document_id"]:
        if frame[col].isna().any():
            raise ValueError(f"В {col} есть пропуски.")
        frame[col] = frame[col].astype(str)
    for col in ["query", "title", "text"]:
        frame[col] = frame[col].fillna("").astype(str)

train["relevance"] = pd.to_numeric(train["relevance"], errors="raise").astype(np.int8)
if not set(train["relevance"].unique()).issubset({0, 1}):
    raise ValueError("Ожидалась бинарная relevance со значениями 0/1.")

sample_submission["id"] = sample_submission["id"].astype(str)
if set(sample_submission["id"]) != set(test["id"]):
    raise ValueError("ID в sample_submission.csv не совпадают с ID в test.csv.")

if len(train) != 31530 or len(test) != 13507:
    raise ValueError(
        f"Размеры не совпали с зафиксированной версией соревнования: "
        f"train={len(train)}, test={len(test)}; ожидались 31530 и 13507."
    )

print("Train:", train.shape, "| Test:", test.shape)
print("Train queries:", train["query_id"].nunique(), "| Test queries:", test["query_id"].nunique())
print("Labels:", train["relevance"].value_counts().sort_index().to_dict())


## 3. Встроенные оценки Experiment 18 и Experiment 20

Чтобы на запуске требовались только три исходных CSV, в ноутбук встроены:
- OOF-оценки Experiment 18 и Experiment 20 LoRA для обучающей выборки;
- test-оценки тех же двух компонентов.

Оценки привязаны к `id`, а не к порядку строк. Перед использованием проверяется полное совпадение ID с исходными таблицами.


In [ ]:
# Встроенные score cache — только оценки, необходимые финальному ансамблю.
# Дополнительные CSV с предыдущих экспериментов не нужны.
CHAMPION_LORA_OOF_GZ_B64 = (
    "..."
)
CHAMPION_LORA_TEST_GZ_B64 = (
    "..."
)

def read_embedded_gzip_csv(payload, name):
    raw = gzip.decompress(base64.b64decode(payload.encode("ascii")))
    frame = pd.read_csv(io.BytesIO(raw), dtype={"id": "string"})
    if "id" not in frame.columns or not frame["id"].is_unique:
        raise ValueError(f"Встроенный кэш {name} имеет некорректный id.")
    frame["id"] = frame["id"].astype(str)
    return frame

cached_oof = read_embedded_gzip_csv(CHAMPION_LORA_OOF_GZ_B64, "OOF")
cached_test = read_embedded_gzip_csv(CHAMPION_LORA_TEST_GZ_B64, "test")

if set(cached_oof["id"]) != set(train["id"]):
    raise ValueError("ID встроенного OOF-кэша не совпадают с train.csv.")
if set(cached_test["id"]) != set(test["id"]):
    raise ValueError("ID встроенного test-кэша не совпадают с test.csv.")

cached_oof = cached_oof.set_index("id").loc[train["id"]].reset_index()
cached_test = cached_test.set_index("id").loc[test["id"]].reset_index()

champion_oof = pd.to_numeric(cached_oof["exp18_oof"], errors="raise").to_numpy(np.float32)
lora_oof_cached = pd.to_numeric(cached_oof["jina_lora_oof"], errors="raise").to_numpy(np.float32)
champion_test = pd.to_numeric(cached_test["exp18_score"], errors="raise").to_numpy(np.float32)
lora_test_cached = pd.to_numeric(cached_test["jina_lora_score"], errors="raise").to_numpy(np.float32)

for name, values in [
    ("Exp18 OOF", champion_oof), ("LoRA OOF", lora_oof_cached),
    ("Exp18 test", champion_test), ("LoRA test", lora_test_cached),
]:
    if len(values) == 0 or not np.isfinite(values).all():
        raise ValueError(f"{name}: пустые или нечисловые значения.")

print("Embedded OOF rows:", len(cached_oof), "| embedded test rows:", len(cached_test))
print("Score cache aligned by id: OK")


## 4. Подготовка пяти фолдов

Используется та же схема, что в экспериментах 18–22: сначала объединяются связанные `query_id` и `document_id` в connected components, затем `GroupKFold(n_splits=5)`. Один связанный компонент не может попасть одновременно в train и validation одного фолда.


In [ ]:
class UnionFind:
    def __init__(self):
        self.parent = {}
        self.rank = {}

    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
            self.rank[x] = 0
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1

uf = UnionFind()
for query_id, document_id in zip(train["query_id"], train["document_id"]):
    uf.union("q:" + str(query_id), "d:" + str(document_id))

component_roots = [uf.find("q:" + str(q)) for q in train["query_id"]]
groups, _ = pd.factorize(component_roots, sort=True)
if len(np.unique(groups)) < N_SPLITS:
    raise ValueError("Недостаточно независимых connected components для пяти фолдов.")

folds = list(GroupKFold(n_splits=N_SPLITS).split(train, train["relevance"], groups))
fold_id_by_row = np.zeros(len(train), dtype=np.int8)

for fold_id, (tr_idx, va_idx) in enumerate(folds, 1):
    fold_id_by_row[va_idx] = fold_id
    overlap = set(groups[tr_idx]).intersection(set(groups[va_idx]))
    if overlap:
        raise AssertionError(f"Пересечение connected components в fold {fold_id}.")

if np.any(fold_id_by_row == 0):
    raise AssertionError("Не все строки получили fold id.")

fold_summary = pd.DataFrame([
    {
        "fold": fold_id,
        "train_rows": len(tr_idx),
        "valid_rows": len(va_idx),
        "valid_queries": int(train.iloc[va_idx]["query_id"].nunique()),
    }
    for fold_id, (tr_idx, va_idx) in enumerate(folds, 1)
])
display(fold_summary)


## 5. Финальный ансамбль и submission

Для каждого запроса оценки двух компонентов переводятся в percentile rank. Финальный score:

\[
S = 0.39\,R_{\text{Exp18}} + 0.61\,R_{\text{Jina-LoRA}}.
\]

Порядок строк итогового `submission.csv` берётся из `sample_submission.csv`. Схема файла — строго `id,prediction`.


In [ ]:
def query_percentile_rank(frame, scores):
    values = pd.Series(np.asarray(scores, dtype=np.float64), index=np.arange(len(frame)))
    query_ids = frame["query_id"].astype(str).reset_index(drop=True)
    return values.groupby(query_ids, sort=False).rank(method="average", pct=True).to_numpy(np.float32)

def dcg_at_k(relevance, k=10):
    relevance = np.asarray(relevance, dtype=np.float64)[:k]
    if len(relevance) == 0:
        return 0.0
    gains = np.power(2.0, relevance) - 1.0
    discounts = np.log2(np.arange(2, len(relevance) + 2))
    return float(np.sum(gains / discounts))

def ndcg10(frame, scores):
    tmp = pd.DataFrame({
        "query_id": frame["query_id"].astype(str).to_numpy(),
        "relevance": frame["relevance"].to_numpy(dtype=np.float64),
        "score": np.asarray(scores, dtype=np.float64),
    })
    values = []
    for _, group in tmp.groupby("query_id", sort=False):
        ideal = np.sort(group["relevance"].to_numpy())[::-1]
        ranked = group.sort_values("score", ascending=False, kind="mergesort")["relevance"].to_numpy()
        idcg = dcg_at_k(ideal, TOP_K)
        values.append(dcg_at_k(ranked, TOP_K) / idcg if idcg > 0 else 0.0)
    return float(np.mean(values))


## 6. Полное повторное обучение Jina v2 + LoRA (опционально)

Если `USE_CACHED_FINAL_SCORES = False`, ноутбук заново обучает пять независимых LoRA-моделей на тех же фолдах и с теми же параметрами Experiment 20:
- `jinaai/jina-reranker-v2-base-multilingual` (278M);
- 512 токенов;
- 6 000 positive/negative пар на фолд;
- 1 эпоха, batch size 4, gradient accumulation 2;
- pairwise softplus loss;
- LoRA: r=8, alpha=16, dropout=0.05;
- target modules: `mixer.out_proj`, `fc1`, `fc2`;
- FP16 на T4.

Этот режим требует GPU и доступа к Hugging Face. Завершённые фолды и Trainer checkpoints сохраняются для возобновления.


In [ ]:
def train_lora_fivefold(train, test, folds, output_dir):
    import torch
    import peft
    import transformers
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        TrainingArguments,
        Trainer,
        DataCollatorWithPadding,
    )
    from torch.utils.data import Dataset

    if not torch.cuda.is_available():
        raise RuntimeError("Для полного переобучения нужен GPU. В Colab выбери T4 GPU.")

    # Совместимость Jina remote code с Transformers 5.22.x.
    import transformers.models.xlm_roberta.modeling_xlm_roberta as xlm_roberta_modeling
    if not hasattr(xlm_roberta_modeling, "create_position_ids_from_input_ids"):
        xlm_roberta_modeling.create_position_ids_from_input_ids = (
            xlm_roberta_modeling.XLMRobertaEmbeddings.create_position_ids_from_input_ids
        )

    # PEFT shim для необязательного torchao в некоторых сборках Colab.
    import peft.import_utils as peft_import_utils
    peft_import_utils.is_torchao_available = lambda: False
    for module_name, module in list(sys.modules.items()):
        if module_name.startswith("peft") and hasattr(module, "is_torchao_available"):
            module.is_torchao_available = lambda: False

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

    def load_base_model():
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_ID,
            trust_remote_code=True,
            torch_dtype="float16",
            use_flash_attn=False,
        )
        model.config.use_cache = False
        return model

    def attach_lora(model):
        linear_names = [
            name for name, module in model.named_modules()
            if isinstance(module, torch.nn.Linear)
        ]
        missing = [
            target for target in LORA_TARGET_MODULES
            if not any(name == target or name.endswith("." + target) for name in linear_names)
        ]
        if missing:
            raise RuntimeError(f"Не найдены LoRA target modules: {missing}")

        config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGET_MODULES,
            modules_to_save=["classifier"],
            bias="none",
        )
        model = get_peft_model(model, config)
        model = model.to("cuda")

        for parameter in model.parameters():
            if parameter.requires_grad and parameter.dtype != torch.float32:
                parameter.data = parameter.data.float()
        trainable_dtypes = {
            str(parameter.dtype) for parameter in model.parameters() if parameter.requires_grad
        }
        if trainable_dtypes != {"torch.float32"}:
            raise TypeError(f"Trainable LoRA parameters must be FP32, got {trainable_dtypes}")
        return model

    def score_frame(model, frame, batch_size=8):
        model.eval()
        queries = frame["query"].fillna("").astype(str).tolist()
        documents = (
            frame["title"].fillna("").astype(str)
            + "\n"
            + frame["text"].fillna("").astype(str)
        ).tolist()
        scores = []
        for start in range(0, len(frame), batch_size):
            encoded = tokenizer(
                queries[start:start + batch_size],
                documents[start:start + batch_size],
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            encoded = {key: value.to("cuda") for key, value in encoded.items()}
            with torch.inference_mode():
                with torch.autocast("cuda", dtype=torch.float16):
                    logits = model(**encoded).logits.reshape(-1).float()
            scores.extend(logits.detach().cpu().numpy().tolist())
        result = np.asarray(scores, dtype=np.float32)
        if len(result) != len(frame) or not np.isfinite(result).all():
            raise FloatingPointError("Jina scoring вернул неверную длину или NaN/Inf.")
        return result

    def build_pair_indices(frame, seed):
        rng = np.random.default_rng(seed)
        pairs = []
        labels = frame["relevance"].to_numpy(dtype=np.int64)

        for _, positions in frame.groupby("query_id", sort=False).indices.items():
            idx = np.asarray(positions, dtype=np.int64)
            positive = idx[labels[idx] == 1]
            negative = idx[labels[idx] == 0]
            if len(positive) == 0 or len(negative) == 0:
                continue

            total = len(positive) * len(negative)
            if total <= PAIR_LIMIT_PER_QUERY:
                local_pairs = [
                    (int(pos_idx), int(neg_idx))
                    for pos_idx in positive
                    for neg_idx in negative
                ]
            else:
                chosen = rng.choice(total, size=PAIR_LIMIT_PER_QUERY, replace=False)
                local_pairs = [
                    (
                        int(positive[pair_id // len(negative)]),
                        int(negative[pair_id % len(negative)]),
                    )
                    for pair_id in chosen
                ]
            pairs.extend(local_pairs)

        if not pairs:
            raise ValueError("Не сформированы positive/negative пары.")
        rng.shuffle(pairs)
        if len(pairs) > PAIR_BUDGET_PER_FOLD:
            selected = rng.choice(len(pairs), size=PAIR_BUDGET_PER_FOLD, replace=False)
            pairs = [pairs[index] for index in selected]
        result = np.asarray(pairs, dtype=np.int64)

        if len(result) != PAIR_BUDGET_PER_FOLD:
            raise ValueError(
                f"Ожидалось {PAIR_BUDGET_PER_FOLD} пар, сформировано {len(result)}."
            )
        if not np.all(frame.iloc[result[:, 0]]["relevance"].to_numpy() == 1):
            raise AssertionError("В positive-части пары встретился relevance != 1.")
        if not np.all(frame.iloc[result[:, 1]]["relevance"].to_numpy() == 0):
            raise AssertionError("В negative-части пары встретился relevance != 0.")
        if not np.all(
            frame.iloc[result[:, 0]]["query_id"].to_numpy()
            == frame.iloc[result[:, 1]]["query_id"].to_numpy()
        ):
            raise AssertionError("Positive и negative пары относятся к разным запросам.")
        return result

    class PairwiseDataset(Dataset):
        def __init__(self, frame, pair_indices):
            self.frame = frame.reset_index(drop=True)
            self.pairs = np.asarray(pair_indices, dtype=np.int64)
            self.queries = self.frame["query"].to_numpy()
            self.documents = (
                self.frame["title"].fillna("").astype(str)
                + "\n"
                + self.frame["text"].fillna("").astype(str)
            ).to_numpy()

        def __len__(self):
            return len(self.pairs)

        def __getitem__(self, index):
            pos_idx, neg_idx = self.pairs[index]
            query = self.queries[pos_idx]
            pos_encoded = tokenizer(
                query, self.documents[pos_idx],
                truncation=True, max_length=MAX_LENGTH
            )
            neg_encoded = tokenizer(
                query, self.documents[neg_idx],
                truncation=True, max_length=MAX_LENGTH
            )
            return {"pos": pos_encoded, "neg": neg_encoded}

    class PairwiseCollator:
        def __init__(self):
            self.pad = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

        def __call__(self, features):
            pos = self.pad([item["pos"] for item in features])
            neg = self.pad([item["neg"] for item in features])
            return {
                **{"pos_" + key: value for key, value in pos.items()},
                **{"neg_" + key: value for key, value in neg.items()},
            }

    class PairwiseTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            pos_inputs = {key[4:]: value for key, value in inputs.items() if key.startswith("pos_")}
            neg_inputs = {key[4:]: value for key, value in inputs.items() if key.startswith("neg_")}
            pos_scores = model(**pos_inputs).logits.reshape(-1).float()
            neg_scores = model(**neg_inputs).logits.reshape(-1).float()
            loss = torch.nn.functional.softplus(-(pos_scores - neg_scores)).mean()
            if not torch.isfinite(loss):
                raise FloatingPointError("Pairwise loss стал NaN/Inf.")
            return (loss, {"pos_scores": pos_scores, "neg_scores": neg_scores}) if return_outputs else loss

    def make_training_arguments(fold_dir, fold_seed):
        updates_per_epoch = math.ceil(
            PAIR_BUDGET_PER_FOLD / (TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)
        )
        warmup_steps = max(1, int(math.ceil(updates_per_epoch * 0.05)))
        kwargs = {
            "output_dir": str(fold_dir / "trainer"),
            "num_train_epochs": NUM_EPOCHS,
            "per_device_train_batch_size": TRAIN_BATCH_SIZE,
            "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_steps": warmup_steps,
            "logging_steps": LOGGING_STEPS,
            "save_strategy": "steps",
            "save_steps": SAVE_STEPS,
            "save_total_limit": 2,
            "fp16": True,
            "bf16": False,
            "optim": "adamw_torch",
            "lr_scheduler_type": "linear",
            "max_grad_norm": 1.0,
            "report_to": "none",
            "remove_unused_columns": False,
            "dataloader_num_workers": 0,
            "gradient_checkpointing": True,
            "gradient_checkpointing_kwargs": {"use_reentrant": False},
            "seed": fold_seed,
            "data_seed": fold_seed,
        }
        supported = inspect.signature(TrainingArguments.__init__).parameters
        filtered = {key: value for key, value in kwargs.items() if key in supported}
        if "warmup_steps" not in filtered:
            raise RuntimeError("Transformers TrainingArguments не поддерживает warmup_steps.")
        return TrainingArguments(**filtered)

    # Небольшой preflight: проверяем загрузку модели, scoring, PEFT и один optimizer step.
    probe_model = attach_lora(load_base_model())
    probe_scores = score_frame(probe_model, train.iloc[:4], batch_size=2)
    if len(probe_scores) != 4:
        raise AssertionError("Preflight scoring вернул неверное число строк.")

    preflight_frame = train.iloc[folds[0][0]].reset_index(drop=True)
    preflight_pairs = build_pair_indices(preflight_frame, SEED + 999)[:2]
    preflight_dataset = PairwiseDataset(preflight_frame, preflight_pairs)
    preflight_kwargs = {
        "output_dir": str(output_dir / "_preflight"),
        "max_steps": 1,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 1,
        "learning_rate": LEARNING_RATE,
        "fp16": True,
        "bf16": False,
        "optim": "adamw_torch",
        "logging_strategy": "no",
        "save_strategy": "no",
        "report_to": "none",
        "remove_unused_columns": False,
        "dataloader_num_workers": 0,
        "gradient_checkpointing": True,
        "gradient_checkpointing_kwargs": {"use_reentrant": False},
        "seed": SEED + 999,
    }
    supported = inspect.signature(TrainingArguments.__init__).parameters
    preflight_kwargs = {key: value for key, value in preflight_kwargs.items() if key in supported}
    preflight_trainer = PairwiseTrainer(
        model=probe_model,
        args=TrainingArguments(**preflight_kwargs),
        train_dataset=preflight_dataset,
        data_collator=PairwiseCollator(),
    )
    preflight_trainer.train()
    del preflight_trainer, probe_model, preflight_dataset
    shutil.rmtree(output_dir / "_preflight", ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()
    print("Jina/PEFT/Trainer preflight: OK")

    oof_lora = np.full(len(train), np.nan, dtype=np.float32)
    test_lora_by_fold = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, 1):
        fold_dir = output_dir / f"fold_{fold_id}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        valid_cache = fold_dir / "lora_valid.npy"
        test_cache = fold_dir / "lora_test.npy"
        manifest_path = fold_dir / "fold_manifest.json"

        if valid_cache.exists() and test_cache.exists() and manifest_path.exists():
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            if manifest.get("config_tag") == "exp21_best_pair_jina_v2_lora_512_v1":
                valid_scores = np.load(valid_cache)
                test_scores = np.load(test_cache)
                if len(valid_scores) == len(va_idx) and len(test_scores) == len(test):
                    oof_lora[va_idx] = valid_scores
                    test_lora_by_fold.append(test_scores)
                    print(f"Fold {fold_id}: restored saved predictions.")
                    continue

        print(f"\n========== FOLD {fold_id}/{N_SPLITS} ==========")
        train_fold = train.iloc[tr_idx].reset_index(drop=True)
        valid_fold = train.iloc[va_idx].reset_index(drop=True)
        pair_indices = build_pair_indices(train_fold, SEED + fold_id)

        model = attach_lora(load_base_model())
        dataset = PairwiseDataset(train_fold, pair_indices)
        trainer = PairwiseTrainer(
            model=model,
            args=make_training_arguments(fold_dir, SEED + fold_id),
            train_dataset=dataset,
            data_collator=PairwiseCollator(),
        )

        checkpoints = sorted(
            [path for path in (fold_dir / "trainer").glob("checkpoint-*") if path.is_dir()],
            key=lambda path: int(path.name.split("-")[-1]),
        )
        resume_checkpoint = str(checkpoints[-1]) if checkpoints else None
        trainer.train(resume_from_checkpoint=resume_checkpoint)

        adapter_dir = fold_dir / "lora_adapter"
        trainer.model.save_pretrained(adapter_dir)
        tokenizer.save_pretrained(adapter_dir)

        valid_scores = score_frame(trainer.model, valid_fold, batch_size=8)
        test_scores = score_frame(trainer.model, test, batch_size=8)
        np.save(valid_cache, valid_scores)
        np.save(test_cache, test_scores)
        manifest_path.write_text(
            json.dumps(
                {
                    "config_tag": "exp21_best_pair_jina_v2_lora_512_v1",
                    "fold": fold_id,
                    "pairs": int(len(pair_indices)),
                    "model_id": MODEL_ID,
                    "max_length": MAX_LENGTH,
                    "epochs": NUM_EPOCHS,
                    "target_modules": LORA_TARGET_MODULES,
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

        oof_lora[va_idx] = valid_scores
        test_lora_by_fold.append(test_scores)

        del trainer, model, dataset
        gc.collect()
        torch.cuda.empty_cache()

    if not np.isfinite(oof_lora).all():
        raise ValueError("После пяти фолдов в OOF остались NaN/Inf.")
    if len(test_lora_by_fold) != N_SPLITS:
        raise ValueError("Не все test-fold predictions получены.")

    test_lora = np.mean(np.vstack(test_lora_by_fold), axis=0).astype(np.float32)
    return oof_lora, test_lora


In [ ]:
if USE_CACHED_FINAL_SCORES:
    print("Режим: встроенные проверенные OOF/test scores; повторное GPU-обучение пропущено.")
    lora_oof = lora_oof_cached
    lora_test = lora_test_cached
else:
    print("Режим: полное пятифолдовочное обучение Jina v2 + LoRA.")
    lora_oof, lora_test = train_lora_fivefold(train, test, folds, OUTPUT_DIR)

exp18_oof_rank = query_percentile_rank(train, champion_oof)
lora_oof_rank = query_percentile_rank(train, lora_oof)
final_oof_scores = FINAL_W_EXP18 * exp18_oof_rank + FINAL_W_LORA * lora_oof_rank
final_oof_ndcg10 = ndcg10(train, final_oof_scores)

exp18_test_rank = query_percentile_rank(test, champion_test)
lora_test_rank = query_percentile_rank(test, lora_test)
final_test_scores = FINAL_W_EXP18 * exp18_test_rank + FINAL_W_LORA * lora_test_rank

print("Final blend weights:", {"Experiment 18": FINAL_W_EXP18, "Jina v2 LoRA": FINAL_W_LORA})
print("OOF NDCG@10:", round(final_oof_ndcg10, 6))
print("Reference Experiment 21 OOF NDCG@10: 0.960996")


## 7. Выход

После выполнения ноутбук создаёт один итоговый файл — `submission.csv` — и проверяет:
- совпадение набора ID с `sample_submission.csv`;
- отсутствие повторяющихся ID;
- отсутствие NaN/Inf в `prediction`;
- точное количество строк.


In [ ]:
test_scores_by_id = pd.Series(
    final_test_scores,
    index=test["id"].astype(str),
    dtype=np.float64,
)
submission = pd.DataFrame({"id": sample_submission["id"].astype(str)})
submission["prediction"] = submission["id"].map(test_scores_by_id)

if submission["prediction"].isna().any():
    missing_ids = submission.loc[submission["prediction"].isna(), "id"].head(10).tolist()
    raise ValueError(f"Для некоторых ID нет prediction: {missing_ids}")

submission["prediction"] = pd.to_numeric(submission["prediction"], errors="raise").astype(np.float64)

if len(submission) != len(test):
    raise AssertionError("Число строк submission не совпадает с test.")
if not submission["id"].is_unique:
    raise AssertionError("В submission повторяются ID.")
if not np.isfinite(submission["prediction"].to_numpy()).all():
    raise FloatingPointError("В submission есть NaN/Inf.")

SUBMISSION_PATH = Path("/content/submission.csv") if Path("/content").is_dir() else Path.cwd() / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

try:
    freeze = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True,
        text=True,
        check=True,
    )
    (OUTPUT_DIR / "requirements_runtime.txt").write_text(freeze.stdout, encoding="utf-8")
except Exception as exc:
    print("Не удалось сохранить pip freeze:", repr(exc))

runtime_manifest = {
    "solution": "Experiment 21 Best Pair",
    "system": platform.system(),
    "platform": platform.platform(),
    "python": sys.version,
    "cached_scores": bool(USE_CACHED_FINAL_SCORES),
    "model_id": MODEL_ID,
    "weights": {"experiment18": FINAL_W_EXP18, "jina_v2_lora": FINAL_W_LORA},
    "max_length": MAX_LENGTH,
    "n_splits": N_SPLITS,
    "output": str(SUBMISSION_PATH),
}
(OUTPUT_DIR / "solution_manifest.json").write_text(
    json.dumps(runtime_manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Submission saved:", SUBMISSION_PATH)
print("Rows:", len(submission), "| Columns:", list(submission.columns))
display(submission.head())
